# 02 — Enriquecimento externo e engenharia de atributos

**Tech Challenge Fase 3 — FIAP Pós-Tech IA Scientist**

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # permite `import src` a partir de notebooks/
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src import config as C
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

## Fontes externas e regra temporal

Todas as fontes são **anteriores ou contemporâneas a 2023**. O script `src/data/download_external.py` baixa, padroniza e consolida; este notebook documenta a cobertura e as decisões de junção.

In [ ]:
ext = pd.read_parquet(C.EXTERNAL_DIR / "socioeconomico_municipal.parquet")
print(ext.shape)
(ext.drop(columns=C.COL_ID).notna().mean() * 100).round(1).sort_values().to_frame("% cobertura")

## Construção do dataset t → t+1

In [ ]:
from src.data import build_dataset
dados = build_dataset.construir()
pd.read_csv(C.PROCESSED_DIR / "dicionario_dataset.csv")

## Tratamento de data leakage — checklist auditável

1. Do ano t+1 entra apenas `em_risco` (e `taxa_alfabetizacao_t1` para análise) — `COLUNAS_PROIBIDAS` em `build_dataset.py`.
2. Toda feature do bloco A tem sufixo `_t`; `features.auditar()` falha se algo com `_t1` chegar a X.
3. Imputação, escala e one-hot vivem dentro do `Pipeline` → ajustados só no treino (`pipeline.py`).
4. `taxa_vs_mediana_uf_t` usa apenas dados de t.
5. Nenhum encoding ordinal "por desempenho" (o `regiao_encoded` da versão anterior foi removido).

In [ ]:
from src.preprocessing.features import separar_xy, tipos
X, y = separar_xy(dados); num, cat = tipos(X)
print(f"{X.shape[1]} features: {len(num)} numéricas, {len(cat)} categóricas ({cat})")

## Sensibilidade do corte do target

In [ ]:
for corte in (50, 55, 60, 65, 70):
    print(f"corte {corte}%: risco = {(dados['taxa_alfabetizacao_t1'] < corte).mean():.1%}")
if f"{C.COL_META_2030}_t" in dados:
    alt = dados["taxa_alfabetizacao_t1"] < 0.75 * dados[f"{C.COL_META_2030}_t"]
    print(f"corte alternativo (75% da meta municipal): risco = {alt.mean():.1%}; concordância com corte 60%: {(alt == dados[C.TARGET].astype(bool)).mean():.1%}")